In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Find all race days

In [2]:
url = "https://racing.hkjc.com/en-us/local/information/localresults"
response = requests.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text)

In [3]:
race_dates = [o.get_text(strip=True) for o in soup.select("#selectId option")]

# Find races for each day

In [4]:
def list_races(date):
    url = f"https://racing.hkjc.com/en-us/local/information/localresults?racedate={date}"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text)

    races = [a["href"] for a in soup.select('div.top_races a[href*="localresults"]')]
    if len(races) < 1:
        return []
    races.append(races[0][:-1] + "1") # a link is not generated for the currently selected (first) race
    return races

In [5]:
list_races("22/04/2026")

['/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=2',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=3',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=4',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=5',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=6',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=7',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=8',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=9',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=1']

# Extract race info

In [6]:
url = "https://racing.hkjc.com/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=2"
response = requests.get(url, allow_redirects=False)
response.raise_for_status()

In [7]:
soup = BeautifulSoup(response.text)

In [8]:
[tuple(map(int, [d.get_text(strip=True) for d in s.select("div>div")])) for s in soup.select("div.performance table.draggable tbody tr td:nth-child(10)")]

[(9, 9, 9, 1),
 (4, 3, 3, 2),
 (8, 8, 8, 3),
 (5, 5, 5, 4),
 (10, 10, 10, 5),
 (11, 11, 11, 6),
 (1, 1, 1, 7),
 (7, 7, 7, 8),
 (6, 6, 6, 9),
 (2, 2, 2, 10),
 (3, 4, 4, 11),
 ()]

In [71]:
def get_race_info(page):
    url = f"https://racing.hkjc.com{page}"
    response = requests.get(url, allow_redirects=False)
    response.raise_for_status()
    if response.status_code != 200:
        return None, None
    soup = BeautifulSoup(response.text)

    date = pd.to_datetime(soup.select_one("#selectId option[selected]").get_text(strip=True), format="%d/%m/%Y")
    
    race_txt = soup.select_one("div.race_tab thead td").get_text(strip=True)
    race_id = int(str(date.year) + race_txt.split("(")[1][:-1].rjust(5, "0"))
    race_no = int(race_txt.split()[1])


    # get general race info
    table = soup.select("div.race_tab tbody td")
    race = {
        "race_id": race_id,
        "race_no": race_no,
        "going": table[5].get_text(strip=True),
        "name": table[6].get_text(strip=True),
        "course": table[8].get_text(strip=True),
        "prize": int("".join(table[9].get_text(strip=True)[4:].split(","))),
        "event_date": date,
        "location":soup.select_one("p.f_clear span").get_text(strip=True).split("  ")[-1]
    }

    def get_column(n, text=True):
        s = soup.select(f"div.performance table.draggable tbody tr td:nth-child({n})")
        if text:
            s = [x.get_text(strip=True) for x in s]
        return s

    horses = get_column(3)
    hnames = [x.split("(")[0].strip() for x in horses]
    hids = [x.split("(")[1][:-1] for x in horses]
    
    runners = {
        "place": get_column(1),
        "selection_no": get_column(2),
        "selection_name": hnames,
        "selection_id": hids,
        "jockey": get_column(4),
        "trainer": get_column(5),
        "actual_weight": get_column(6),
        "declared_weight": get_column(7),
        "draw": get_column(8),
        "lbw": get_column(9),
        "running_position": [tuple(map(int, [d.get_text(strip=True) for d in s.select("div>div")])) for s in get_column(10, False)],
        "finish_time": get_column(11),
        "bsp": get_column(12)
    }
    runners = pd.DataFrame(runners).assign(race_id=race_id)

    return race, runners

In [72]:
get_race_info("/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=2")[1]

,place,selection_no,selection_name,selection_id,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id
0,1,7,PODIUM,H173,C L Chau,D Eustace,126,1055,1,---,"(9, 9, 9, 1)",1:39.66,5,202600626
1,2,2,SOARING BRONCO,J162,J Orman,M Newnham,133,1123,9,3/4,"(4, 3, 3, 2)",1:39.77,2.7,202600626
2,3,11,SMILING EMPEROR,J371,M L Yeung,K H Ting,126,1179,3,2-1/4,"(8, 8, 8, 3)",1:40.01,17,202600626
3,4,8,VERBIER,J187,J Moreira,C Fownes,128,1242,2,4-1/4,"(5, 5, 5, 4)",1:40.32,3.7,202600626
4,5,3,NOBLE FANS,K038,R Kingscote,J Size,132,1139,7,4-3/4,"(10, 10, 10, 5)",1:40.40,28,202600626
5,6,12,CASA LEGEND,G126,M Chadwick,C W Chang,115,1157,11,6-1/4,"(11, 11, 11, 6)",1:40.64,23,202600626
6,7,1,LUCKY BLESSING,K073,E C W Wong,W K Mo,130,1076,12,6-1/4,"(1, 1, 1, 7)",1:40.66,67,202600626
7,8,5,SPARKLE AND GOLD,K259,B Avdulla,C H Yip,130,1149,5,7-1/2,"(7, 7, 7, 8)",1:40.84,26,202600626
8,9,9,DOUBLE SHOW,G250,K Teetan,B Crawford,127,1211,6,11-1/4,"(6, 6, 6, 9)",1:41.45,20,202600626
9,10,4,TELECOM POWER,J332,E Brown,Y S Tsui,132,1070,10,12-1/2,"(2, 2, 2, 10)",1:41.67,9.7,202600626


# All together

In [11]:
d = race_dates[0]

In [12]:
races = list_races(d)
races

['/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=1',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=2',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=3',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=4',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=5',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=6',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=7',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=8',
 '/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=1']

In [13]:
get_race_info(races[0])

({'race_id': 745,
  'race_no': 1,
  'going': 'GOOD TO FIRM',
  'name': 'MOUNT BUTLER HANDICAP',
  'course': 'TURF - "C" Course',
  'prize': 875000,
  'date': '03/06/2026',
  'location': 'Happy Valley'},
    place selection_no      selection_name selection_id      jockey  \
 0      1            3      FAMILY FORTUNE                J Moreira   
 1      2           12      THE WAY WE WIN                   H T Mo   
 2      3            6       TELECOM POWER                H Bentley   
 3      4            2             SETANTA               L Ferraris   
 4      5            9     WAH MAY WAI WAI                 K Teetan   
 5      6            8  MANYTHANKS FOREVER                 A Atzeni   
 6      7           11         CASA LEGEND                 M F Poon   
 7      8            4         PRINCE ALEX                B Avdulla   
 8      9            7         ZETTA FORCE                 C L Chau   
 9     10           10          SMART TRIO               L Hewitson   
 10    11       

In [14]:
all_races = []
all_runners = []

In [15]:
start_idx=0
if len(all_races)>0:
    start_idx = race_dates.index(all_races[-1]["date"])

for d in race_dates[start_idx:]:
    print(d)
    races = list_races(d)
    for r in races:
        print(r)
        try:
            race, runners = get_race_info(r)
            if race:
                all_races.append(race)
                all_runners.append(runners)
        except:
            continue

03/06/2026
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=1
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=2
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=3
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=4
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=5
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=6
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=7
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=8
/en-us/local/information/localresults?racedate=2026/06/03&Racecourse=HV&RaceNo=1
31/05/2026
/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=2
/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=3
/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=4
/en-us

In [16]:
runners = pd.concat(all_runners, ignore_index=True)

In [17]:
races = pd.DataFrame(all_races)

In [23]:
runners

,place,selection_no,selection_name,selection_id,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id
0,1,3,FAMILY FORTUNE,,J Moreira,C Fownes,133,995,1,---,"(7, 7, 6, 1)",1:38.85,2.7,745
1,2,12,THE WAY WE WIN,,H T Mo,J Richards,113,1076,6,N,"(1, 1, 1, 2)",1:38.90,38,745
2,3,6,TELECOM POWER,,H Bentley,Y S Tsui,130,1053,10,1-1/2,"(11, 11, 10, 3)",1:39.09,8.3,745
3,4,2,SETANTA,,L Ferraris,M Newnham,134,1178,9,2-1/4,"(9, 9, 9, 4)",1:39.19,7.4,745
4,5,9,WAH MAY WAI WAI,,K Teetan,A S Cruz,119,1056,2,2-1/4,"(4, 4, 3, 5)",1:39.20,3.5,745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19680,9,7,KINGLY DEMEANOR,,B Avdulla,A S Cruz,130,1158,9,6,"(6, 7, 9, 9)",1:36.02,17,1
19681,10,13,HAPPY HUNK,,K C Leung,K L Man,119,1208,7,7-1/2,"(2, 2, 3, 10)",1:36.29,24,1
19682,11,3,VERBIER,,E C W Wong,C Fownes,128,1242,2,10-1/4,"(1, 1, 1, 11)",1:36.73,30,1
19683,12,12,LEATHER MASTER,,B Thompson,T P Yung,122,1122,3,11-1/4,"(5, 5, 7, 12)",1:36.86,11,1


In [24]:
races

,race_id,race_no,going,name,course,prize,date,location
0,745,1,GOOD TO FIRM,MOUNT BUTLER HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,Happy Valley
1,746,2,GOOD TO FIRM,MOUNT NICHOLSON HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,Happy Valley
2,747,3,GOOD TO FIRM,WONG NAI CHUNG GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley
3,748,4,GOOD TO FIRM,MIDDLE GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley
4,749,5,GOOD TO FIRM,THE CRICKET CLUB VALLEY STAKES (HANDICAP),"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley
...,...,...,...,...,...,...,...,...
1571,7,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,08/09/2024,Sha Tin
1572,8,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin
1573,9,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin
1574,10,10,GOOD,SUNSET PEAK HANDICAP,"TURF - ""A"" Course",1860000,08/09/2024,Sha Tin


# Normalise Data

In [214]:
races

,race_id,race_no,going,name,course,prize,date,location,event_id
0,745,1,GOOD TO FIRM,MOUNT BUTLER HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,Happy Valley,13
1,746,2,GOOD TO FIRM,MOUNT NICHOLSON HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,Happy Valley,13
2,747,3,GOOD TO FIRM,WONG NAI CHUNG GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley,13
3,748,4,GOOD TO FIRM,MIDDLE GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley,13
4,749,5,GOOD TO FIRM,THE CRICKET CLUB VALLEY STAKES (HANDICAP),"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley,13
...,...,...,...,...,...,...,...,...,...
1571,7,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,08/09/2024,Sha Tin,42
1572,8,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin,42
1573,9,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin,42
1574,10,10,GOOD,SUNSET PEAK HANDICAP,"TURF - ""A"" Course",1860000,08/09/2024,Sha Tin,42


In [212]:
"""new_ids = pd.to_datetime(races.date, format="%d/%m/%Y").dt.year.astype(str) + races.race_id.astype(str).str.rjust(5, "0")
new_ids"""

0       202600745
1       202600746
2       202600747
3       202600748
4       202600749
          ...    
1571    202400007
1572    202400008
1573    202400009
1574    202400010
1575    202400001
Length: 1576, dtype: str

In [139]:
runners

,place,selection_no,selection_name,selection_id,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id
0,1,3,FAMILY FORTUNE,,J Moreira,C Fownes,133,995,1,---,"(7, 7, 6, 1)",1:38.85,2.7,745
1,2,12,THE WAY WE WIN,,H T Mo,J Richards,113,1076,6,N,"(1, 1, 1, 2)",1:38.90,38,745
2,3,6,TELECOM POWER,,H Bentley,Y S Tsui,130,1053,10,1-1/2,"(11, 11, 10, 3)",1:39.09,8.3,745
3,4,2,SETANTA,,L Ferraris,M Newnham,134,1178,9,2-1/4,"(9, 9, 9, 4)",1:39.19,7.4,745
4,5,9,WAH MAY WAI WAI,,K Teetan,A S Cruz,119,1056,2,2-1/4,"(4, 4, 3, 5)",1:39.20,3.5,745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19680,9,7,KINGLY DEMEANOR,,B Avdulla,A S Cruz,130,1158,9,6,"(6, 7, 9, 9)",1:36.02,17,1
19681,10,13,HAPPY HUNK,,K C Leung,K L Man,119,1208,7,7-1/2,"(2, 2, 3, 10)",1:36.29,24,1
19682,11,3,VERBIER,,E C W Wong,C Fownes,128,1242,2,10-1/4,"(1, 1, 1, 11)",1:36.73,30,1
19683,12,12,LEATHER MASTER,,B Thompson,T P Yung,122,1122,3,11-1/4,"(5, 5, 7, 12)",1:36.86,11,1


In [166]:
"""n_runners = runners.assign(race_id=runners.apply(
    lambda r: new_ids[races.race_id == r.race_id].iloc[0 if r.name <= 9372 else -1],
    axis=1
))
n_runners"""

,place,selection_no,selection_name,selection_id,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id
0,1,3,FAMILY FORTUNE,,J Moreira,C Fownes,133,995,1,---,"(7, 7, 6, 1)",1:38.85,2.7,202600745
1,2,12,THE WAY WE WIN,,H T Mo,J Richards,113,1076,6,N,"(1, 1, 1, 2)",1:38.90,38,202600745
2,3,6,TELECOM POWER,,H Bentley,Y S Tsui,130,1053,10,1-1/2,"(11, 11, 10, 3)",1:39.09,8.3,202600745
3,4,2,SETANTA,,L Ferraris,M Newnham,134,1178,9,2-1/4,"(9, 9, 9, 4)",1:39.19,7.4,202600745
4,5,9,WAH MAY WAI WAI,,K Teetan,A S Cruz,119,1056,2,2-1/4,"(4, 4, 3, 5)",1:39.20,3.5,202600745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19680,9,7,KINGLY DEMEANOR,,B Avdulla,A S Cruz,130,1158,9,6,"(6, 7, 9, 9)",1:36.02,17,202400001
19681,10,13,HAPPY HUNK,,K C Leung,K L Man,119,1208,7,7-1/2,"(2, 2, 3, 10)",1:36.29,24,202400001
19682,11,3,VERBIER,,E C W Wong,C Fownes,128,1242,2,10-1/4,"(1, 1, 1, 11)",1:36.73,30,202400001
19683,12,12,LEATHER MASTER,,B Thompson,T P Yung,122,1122,3,11-1/4,"(5, 5, 7, 12)",1:36.86,11,202400001


### Implied odds

In [167]:
n_runners = n_runners.assign(implied_odds=1/pd.to_numeric(runners.bsp, errors="coerce"))
n_runners = n_runners.dropna()
n_runners

,place,selection_no,selection_name,selection_id,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id,implied_odds
0,1,3,FAMILY FORTUNE,,J Moreira,C Fownes,133,995,1,---,"(7, 7, 6, 1)",1:38.85,2.7,202600745,0.370370
1,2,12,THE WAY WE WIN,,H T Mo,J Richards,113,1076,6,N,"(1, 1, 1, 2)",1:38.90,38,202600745,0.026316
2,3,6,TELECOM POWER,,H Bentley,Y S Tsui,130,1053,10,1-1/2,"(11, 11, 10, 3)",1:39.09,8.3,202600745,0.120482
3,4,2,SETANTA,,L Ferraris,M Newnham,134,1178,9,2-1/4,"(9, 9, 9, 4)",1:39.19,7.4,202600745,0.135135
4,5,9,WAH MAY WAI WAI,,K Teetan,A S Cruz,119,1056,2,2-1/4,"(4, 4, 3, 5)",1:39.20,3.5,202600745,0.285714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19680,9,7,KINGLY DEMEANOR,,B Avdulla,A S Cruz,130,1158,9,6,"(6, 7, 9, 9)",1:36.02,17,202400001,0.058824
19681,10,13,HAPPY HUNK,,K C Leung,K L Man,119,1208,7,7-1/2,"(2, 2, 3, 10)",1:36.29,24,202400001,0.041667
19682,11,3,VERBIER,,E C W Wong,C Fownes,128,1242,2,10-1/4,"(1, 1, 1, 11)",1:36.73,30,202400001,0.033333
19683,12,12,LEATHER MASTER,,B Thompson,T P Yung,122,1122,3,11-1/4,"(5, 5, 7, 12)",1:36.86,11,202400001,0.090909


### Win/Lose

In [168]:
n_runners = n_runners.assign(win_lose=n_runners.place=="1")

### Finish Time

In [169]:
n_runners.loc[n_runners.finish_time == "---", "finish_time"] = "00:00.00"
n_runners.finish_time = "00:" + n_runners.finish_time

In [170]:
n_runners.finish_time.sort_values()

2955    00:00:00.00
7110    00:00:00.00
7760    00:00:00.00
7829    00:00:00.00
9023    00:00:00.00
           ...     
1154     00:2:31.55
1155     00:2:31.65
1156     00:2:31.79
1157     00:2:33.17
6160     00:2:35.56
Name: finish_time, Length: 19427, dtype: str

### Types

In [171]:
n_runners = n_runners.astype({
    "place": "string",
    "selection_no": "int8",
    "selection_name": "string",
    "selection_id": "string",
    "jockey": "string",
    "trainer": "string",
    "actual_weight": "int16",
    "declared_weight": "int16",
    "draw": "int8",
    "lbw": "string",
    #"running_position": "string",
    "finish_time": "timedelta64[s]",
    "bsp": "float64",
    "race_id": "int64",
    "implied_odds": "float64",
    "win_lose": "bool"
})

## Races

In [172]:
races

,race_id,race_no,going,name,course,prize,date,location
0,745,1,GOOD TO FIRM,MOUNT BUTLER HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,Happy Valley
1,746,2,GOOD TO FIRM,MOUNT NICHOLSON HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,Happy Valley
2,747,3,GOOD TO FIRM,WONG NAI CHUNG GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley
3,748,4,GOOD TO FIRM,MIDDLE GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley
4,749,5,GOOD TO FIRM,THE CRICKET CLUB VALLEY STAKES (HANDICAP),"TURF - ""C"" Course",1170000,03/06/2026,Happy Valley
...,...,...,...,...,...,...,...,...
1571,7,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,08/09/2024,Sha Tin
1572,8,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin
1573,9,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin
1574,10,10,GOOD,SUNSET PEAK HANDICAP,"TURF - ""A"" Course",1860000,08/09/2024,Sha Tin


### Create Events

In [245]:
#n_races = races.assign(race_id=new_ids).drop_duplicates()#.set_index("race_id")
n_races = races
n_races["event_id"] = n_races.groupby(["date", "location"]).ngroup()
n_events = n_races[["event_date", "location", "event_id"]].drop_duplicates().set_index("event_id")
n_races = n_races.drop(columns=["event_date", "location"])

In [246]:
n_events

,event_date,location
event_id,,
13,03/06/2026,Happy Valley
164,31/05/2026,Sha Tin
145,27/05/2026,Happy Valley
129,24/05/2026,Sha Tin
108,20/05/2026,Happy Valley
...,...,...
120,22/09/2024,Sha Tin
98,18/09/2024,Happy Valley
83,15/09/2024,Sha Tin


In [247]:
n_races

,race_id,race_no,going,name,course,prize,date,event_id
0,202600745,1,GOOD TO FIRM,MOUNT BUTLER HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,13
1,202600746,2,GOOD TO FIRM,MOUNT NICHOLSON HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,13
2,202600747,3,GOOD TO FIRM,WONG NAI CHUNG GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,13
3,202600748,4,GOOD TO FIRM,MIDDLE GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,13
4,202600749,5,GOOD TO FIRM,THE CRICKET CLUB VALLEY STAKES (HANDICAP),"TURF - ""C"" Course",1170000,03/06/2026,13
...,...,...,...,...,...,...,...,...
1571,202400007,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,08/09/2024,42
1572,202400008,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,42
1573,202400009,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,42
1574,202400010,10,GOOD,SUNSET PEAK HANDICAP,"TURF - ""A"" Course",1860000,08/09/2024,42


### Types

In [248]:
n_races = n_races.astype({
    "race_id": "int64",
    "race_no": "int8",
    "going": "string",
    "name": "string",
    "course": "string",
    "prize": "int64",
    #"date": "datetime64[s]",
    #"location": "string"
}).set_index("race_id")
n_events = n_events.astype({
    "event_date": "datetime64[s]",
    "location": "string"
})

## Observe

In [249]:
n_runners

,place,selection_no,selection_name,selection_id,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id,implied_odds,win_lose
0,1,3,FAMILY FORTUNE,,J Moreira,C Fownes,133,995,1,---,"(7, 7, 6, 1)",0 days 00:01:38,2.7,202600745,0.370370,True
1,2,12,THE WAY WE WIN,,H T Mo,J Richards,113,1076,6,N,"(1, 1, 1, 2)",0 days 00:01:38,38.0,202600745,0.026316,False
2,3,6,TELECOM POWER,,H Bentley,Y S Tsui,130,1053,10,1-1/2,"(11, 11, 10, 3)",0 days 00:01:39,8.3,202600745,0.120482,False
3,4,2,SETANTA,,L Ferraris,M Newnham,134,1178,9,2-1/4,"(9, 9, 9, 4)",0 days 00:01:39,7.4,202600745,0.135135,False
4,5,9,WAH MAY WAI WAI,,K Teetan,A S Cruz,119,1056,2,2-1/4,"(4, 4, 3, 5)",0 days 00:01:39,3.5,202600745,0.285714,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19680,9,7,KINGLY DEMEANOR,,B Avdulla,A S Cruz,130,1158,9,6,"(6, 7, 9, 9)",0 days 00:01:36,17.0,202400001,0.058824,False
19681,10,13,HAPPY HUNK,,K C Leung,K L Man,119,1208,7,7-1/2,"(2, 2, 3, 10)",0 days 00:01:36,24.0,202400001,0.041667,False
19682,11,3,VERBIER,,E C W Wong,C Fownes,128,1242,2,10-1/4,"(1, 1, 1, 11)",0 days 00:01:36,30.0,202400001,0.033333,False
19683,12,12,LEATHER MASTER,,B Thompson,T P Yung,122,1122,3,11-1/4,"(5, 5, 7, 12)",0 days 00:01:36,11.0,202400001,0.090909,False


In [250]:
n_races

,race_no,going,name,course,prize,date,event_id
race_id,,,,,,,
202600745,1,GOOD TO FIRM,MOUNT BUTLER HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,13
202600746,2,GOOD TO FIRM,MOUNT NICHOLSON HANDICAP,"TURF - ""C"" Course",875000,03/06/2026,13
202600747,3,GOOD TO FIRM,WONG NAI CHUNG GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,13
202600748,4,GOOD TO FIRM,MIDDLE GAP HANDICAP,"TURF - ""C"" Course",1170000,03/06/2026,13
202600749,5,GOOD TO FIRM,THE CRICKET CLUB VALLEY STAKES (HANDICAP),"TURF - ""C"" Course",1170000,03/06/2026,13
...,...,...,...,...,...,...,...
202400007,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,08/09/2024,42
202400008,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,42
202400009,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,42


In [251]:
n_events

,event_date,location
event_id,,
13,2026-03-06,Happy Valley
164,2026-05-31,Sha Tin
145,2026-05-27,Happy Valley
129,2026-05-24,Sha Tin
108,2026-05-20,Happy Valley
...,...,...
120,2024-09-22,Sha Tin
98,2024-09-18,Happy Valley
83,2024-09-15,Sha Tin


# Store

In [252]:
n_races.to_parquet("data/hkjc-races.parquet")
n_runners.to_parquet("data/hkjc-runners.parquet")
n_events.to_parquet("data/hkjc-events.parquet")